In [0]:
# ======================================
# Config (reuse from 1_topic_tagging_v5)
# ======================================

from __future__ import annotations
import json, re, time
from typing import Any, Dict, List
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

from mlflow.deployments import get_deploy_client

#sandbox.z_jungryo_lee.category_topic_detail_merged_category_topic_dynamic_rules_allgroups_v1_merge_v1 LLM endpoints
FULL_ENDPOINT = "databricks-gpt-5-4"
MINI_ENDPOINT = "databricks-gpt-5-4-mini"

# DB / Tables
PROMPT_VERSION = "category_topic_dynamic_rules_allgroups_v1"
SAVE_DB = "sandbox.z_jungryo_lee"

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"

RULE_PROFILE_TABLE = f"{SAVE_DB}.category_topic_rule_profile_{PROMPT_VERSION}"
TOPIC_POOL_TABLE   = f"{SAVE_DB}.category_topic_pool_{PROMPT_VERSION}"
DETAIL_TABLE       = f"{SAVE_DB}.category_topic_detail_{PROMPT_VERSION}"
SUMMARY_TABLE      = f"{SAVE_DB}.category_topic_summary_{PROMPT_VERSION}"

PROGRESS_TABLE     = f"{SAVE_DB}.category_topic_progress_{PROMPT_VERSION}"

# New helper tables (this notebook only)
REP_EMB_TABLE      = f"{SAVE_DB}.category_topic_representative_embedding_{PROMPT_VERSION}"

# similarity threshold
SIM_THRESHOLD = 0.85


In [0]:
# ======================================
# Common Helpers (from 1_topic_tagging_v5)
# - For resume / hybrid classify notebook
# ======================================


# --------------------------------------------------
# Utils
# --------------------------------------------------
def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


# --------------------------------------------------
# Group sample loader (✅ 기존 1번 py 로직)
# --------------------------------------------------
def load_group_sample_df(
    cate_1_depth: str,
    cate_2_depth: str,
    sc_measurement: int,
    max_rows: int
):
    """
    Deterministic sampling per group (md5 + fixed seed)
    """
    query = f"""
    with base as (
        select distinct memo
        from {SOURCE_TABLE}
        where cate_1_depth = '{clean_text(cate_1_depth)}'
          and cate_2_depth = '{clean_text(cate_2_depth)}'
          and sc_measurement = {int(sc_measurement)}
          and memo is not null
          and length(trim(memo)) > 0
    ),
    sampled as (
        select
            memo,
            row_number() over (
                order by md5(
                    concat(
                        coalesce(memo, ''),
                        '{clean_text(cate_1_depth)}',
                        '{clean_text(cate_2_depth)}',
                        '{int(sc_measurement)}',
                        'seed_20260420'
                    )
                )
            ) as rn
        from base
    )
    select memo
    from sampled
    where rn <= {int(max_rows)}
    order by rn
    """
    return spark.sql(query)


# --------------------------------------------------
# Progress logging helpers
# --------------------------------------------------
PROGRESS_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])


def append_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        df.write.mode("append").format("delta").saveAsTable(table_name)
    else:
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def log_progress(
    cate_1_depth: str,
    cate_2_depth: str,
    sc_measurement: int,
    stage: str,
    status: str,
    message: str = ""
):
    """
    Compatible with 기존 category_topic_progress_ 테이블
    """
    row = [(
        clean_text(cate_1_depth),
        clean_text(cate_2_depth),
        int(sc_measurement),
        clean_text(stage),
        clean_text(status),
        clean_text(message),
        pd.Timestamp.now().to_pydatetime()
    )]
    df = spark.createDataFrame(row, schema=PROGRESS_STRUCT)
    append_table(df, PROGRESS_TABLE)


print("[OK] Common helper functions loaded")


In [0]:
progress_df = spark.table(f"{SAVE_DB}.category_topic_group_status")

target_groups = (
    progress_df
    .where("is_done = 0")
    .select("cate_1_depth", "cate_2_depth", "sc_measurement")
    .distinct()
    .collect()
)

print(f"[INFO] Pending groups count = {len(target_groups)}")

In [0]:
# ======================================
# 3) Overall Candidate Rule (pre-filter)
# - Korean / English / German / Spanish
# - 후보 추리기 전용 (판단은 mini-LLM)
# ======================================

import re

GENERIC_PATTERN_LIST = [
    # ---------- Korean ----------
    r"^좋아요+$",
    r"^별로예요?$",
    r"^만족(합니다|스러워요)?$",
    r"^불만입니다?$",
    r"^최고(예요|입니다)?$",
    r"^대만족$",
    r"^강추$",
    r"^추천합니다?$",
    r"^굿+$",
    r"^괜찮아요$",
    r"^나쁘지 않아요$",

    # ---------- English ----------
    r"^(very\s+)?good$",
    r"^(really\s+)?great$",
    r"^excellent$",
    r"^awesome$",
    r"^perfect$",
    r"^love\s+it$",
    r"^i\s+like\s+it$",
    r"^not\s+bad$",
    r"^okay$",

    # ---------- German ----------
    r"^sehr\s+gut$",
    r"^gut$",
    r"^ausgezeichnet$",
    r"^perfekt$",
    r"^toll$",
    r"^super$",
    r"^nicht\s+schlecht$",
    r"^zufrieden$",

    # ---------- Spanish ----------
    r"^muy\s+bueno$",
    r"^bueno$",
    r"^excelente$",
    r"^perfecto$",
    r"^genial$",
    r"^me\s+gusta$",
    r"^no\s+está\s+mal$",
    r"^satisfecho$",
]

GENERIC_PATTERNS = [re.compile(p, re.IGNORECASE) for p in GENERIC_PATTERN_LIST]

# reason / causality blockers (공통)
REASON_BLOCKER = re.compile(
    r"because|due to|so that|therefore|"
    r"deshalb|weil|"
    r"por|porque|"
    r"그래서|때문",
    re.IGNORECASE
)

def is_overall_candidate_rule(memo: str) -> bool:
    """
    Overall 후보만 선별 (판단 X)
    - 짧은 단문
    - 이유 단서 없음
    - 언어별 제너릭 감탄/평가만 허용
    """
    if memo is None:
        return False

    text = memo.strip().lower()

    # 너무 길면 제외
    if len(text) > 50:
        return False

    # 이유/인과 단서 있으면 제외
    if REASON_BLOCKER.search(text):
        return False

    # 순수 제너릭 감탄만 허용
    return any(p.fullmatch(text) for p in GENERIC_PATTERNS)


print("[OK] Overall candidate rule loaded")

In [0]:
llm_client = get_deploy_client("databricks")

def overall_gate_mini(memo: str) -> bool:
    prompt = f"""
You judge whether a review is only a pure sentiment expression.

If it contains ANY specific reason, feature, function or explanation,
return false.

Review:
{memo}

Return JSON only:
{{"is_overall": true or false}}
"""
    resp = llm_client.predict(
        endpoint=MINI_ENDPOINT,
        inputs={
            "messages": [
                {"role": "system", "content": "You are a strict binary classifier."},
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 50
        }
    )
    return json.loads(resp["choices"][0]["message"]["content"])["is_overall"]


In [0]:
detail_df = spark.table(DETAIL_TABLE)

rep_df = (
    detail_df
    .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic")
    .agg(F.first("memo").alias("rep_memo"))
)

display(rep_df.limit(10))


In [0]:
# ======================================
# 5) Similarity-based Topic Assignment
# - representative_memos_json only
# - overall topics handled separately
# ======================================

import json
import re
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Load merge mapping: overall source topics
# --------------------------------------------------
MERGE_MAPPING_TABLE = (
    "sandbox.z_jungryo_lee."
    "category_topic_merge_mapping_"
    "category_topic_dynamic_rules_allgroups_v1_merge_v1"
)

merge_df = spark.table(MERGE_MAPPING_TABLE)

overall_source_topics_df = (
    merge_df
    .where(F.col("merged_topic").isin("전반적 긍정", "전반적 부정"))
    .select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "source_topic",
        "merged_topic"
    )
    .distinct()
)

# --------------------------------------------------
# 2. Load topic pool (split overall / non-overall)
# --------------------------------------------------
topic_pool_df = spark.table(TOPIC_POOL_TABLE)

# (A) overall 계열 topic (유사도 ≥ 0.9 시 1차 overall로 귀속)
topic_pool_overall_df = (
    topic_pool_df.alias("tp")
    .join(
        overall_source_topics_df.alias("ov"),
        on=[
            F.col("tp.cate_1_depth") == F.col("ov.cate_1_depth"),
            F.col("tp.cate_2_depth") == F.col("ov.cate_2_depth"),
            F.col("tp.sc_measurement") == F.col("ov.sc_measurement"),
            F.col("tp.topic") == F.col("ov.source_topic"),
        ],
        how="inner"
    )
    .select(
        "tp.cate_1_depth",
        "tp.cate_2_depth",
        "tp.sc_measurement",
        "ov.merged_topic",
        "tp.representative_memos_json"
    )
)

# (B) 일반 세부 topic (기존 유사도 매칭용)
topic_pool_non_overall_df = (
    topic_pool_df.alias("tp")
    .join(
        overall_source_topics_df.alias("ov"),
        on=[
            F.col("tp.cate_1_depth") == F.col("ov.cate_1_depth"),
            F.col("tp.cate_2_depth") == F.col("ov.cate_2_depth"),
            F.col("tp.sc_measurement") == F.col("ov.sc_measurement"),
            F.col("tp.topic") == F.col("ov.source_topic"),
        ],
        how="left_anti"
    )
    .select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "topic",
        "representative_memos_json"
    )
)

# --------------------------------------------------
# 3. Token / Jaccard helpers
# --------------------------------------------------
def tokenize(text: str):
    if not text:
        return set()
    text = text.lower()
    text = re.sub(r"[^a-z0-9가-힣\s]", " ", text)
    return set(text.split())

def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

# --------------------------------------------------
# 4. Build representative maps
# --------------------------------------------------
# key = (cate1, cate2, sc)

overall_rep_map = {}
for r in topic_pool_overall_df.collect():
    key = (r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"]))
    reps = json.loads(r["representative_memos_json"] or "[]")
    tokens = [tokenize(m) for m in reps]
    overall_rep_map.setdefault(key, []).append({
        "merged_topic": r["merged_topic"],
        "rep_tokens": tokens
    })

non_overall_rep_map = {}
for r in topic_pool_non_overall_df.collect():
    key = (r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"]))
    reps = json.loads(r["representative_memos_json"] or "[]")
    tokens = [tokenize(m) for m in reps]
    non_overall_rep_map.setdefault(key, []).append({
        "topic": r["topic"],
        "rep_tokens": tokens
    })

# --------------------------------------------------
# 5. Similarity decision functions
# --------------------------------------------------
OVERALL_SIM_THRESHOLD = 0.90
NON_OVERALL_SIM_THRESHOLD = 0.25

def match_overall_by_similarity(memo: str, reps: list):
    memo_tk = tokenize(memo)
    best_topic, best_score = None, 0.0
    for t in reps:
        for rt in t["rep_tokens"]:
            s = jaccard(memo_tk, rt)
            if s > best_score:
                best_score, best_topic = s, t["merged_topic"]
    if best_score >= OVERALL_SIM_THRESHOLD:
        return best_topic
    return None

def match_non_overall_by_similarity(memo: str, reps: list):
    memo_tk = tokenize(memo)
    best_topic, best_score = None, 0.0
    for t in reps:
        for rt in t["rep_tokens"]:
            s = jaccard(memo_tk, rt)
            if s > best_score:
                best_score, best_topic = s, t["topic"]
    if best_score >= NON_OVERALL_SIM_THRESHOLD:
        return best_topic
    return None

print("[OK] Overall-aware similarity matcher ready")


In [0]:
import numpy as np

def cosine(a, b):
    a = np.array(a); b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [0]:
def match_by_similarity(memo_emb, rep_rows):
    best_topic = None
    best_score = 0.0
    for r in rep_rows:
        score = cosine(memo_emb, r["embedding"])
        if score > best_score:
            best_score = score
            best_topic = r["topic"]
    if best_score >= SIM_THRESHOLD:
        return best_topic
    return None

In [0]:
def load_group_sample_df(
    cate_1_depth: str,
    cate_2_depth: str,
    sc_measurement: int,
    max_rows: int
):
    query = f"""
    with base as (
        select distinct memo
        from {SOURCE_TABLE}
        where cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and sc_measurement = {int(sc_measurement)}
          and memo is not null
          and length(trim(memo)) > 0
    ),
    sampled as (
        select
            memo,
            row_number() over (
                order by md5(
                    concat(
                        memo,
                        '{cate_1_depth}',
                        '{cate_2_depth}',
                        '{sc_measurement}',
                        'seed_20260420'
                    )
                )
            ) as rn
        from base
    )
    select memo
    from sampled
    where rn <= {int(max_rows)}
    order by rn
    """
    return spark.sql(query)
``

In [0]:
# ======================================
# Full LLM classifier (fallback only)
# - engine: databricks-gpt-5-4
# - used only if similarity match failed
# ======================================

from mlflow.deployments import get_deploy_client
import json

LLM_CLIENT = get_deploy_client("databricks")
FULL_ENDPOINT = "databricks-gpt-5-4"

def classify_with_full_llm(
    memo: str,
    topic_payload: list
) -> str:
    """
    memo: str
    topic_payload: [{"topic": "..."}]  # non-overall topics only
    return: selected topic (string)
    """

    system_prompt = (
        "You are a VOC review classifier.\n"
        "Choose exactly ONE topic from the given list.\n"
        "Do NOT create new topics.\n"
        "The review has already passed overall/generic filtering.\n"
        "Focus on the specific reason in the review."
    )

    user_prompt = f"""
Available topics:
{json.dumps(topic_payload, ensure_ascii=False)}

Review:
{memo}

Return JSON only:
{{"topic": ""}}
"""

    response = LLM_CLIENT.predict(
        endpoint=FULL_ENDPOINT,
        inputs={
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0,
            "max_tokens": 200
        }
    )

    content = response["choices"][0]["message"]["content"]

    try:
        return json.loads(content)["topic"]
    except Exception:
        # ⚠️ 비정상 응답 시 안전장치
        return None

In [0]:
# ======================================
# 8) Hybrid classify main loop
# - checkpoint / resume safe
# - append-only
# ======================================

from pyspark.sql import functions as F
from pyspark.sql import Window
import json
import time


# --------------------------------------------------
# 0. 이미 완료된 그룹 key 로드 (resume 용)
# --------------------------------------------------
if spark.catalog.tableExists(PROGRESS_TABLE):
    done_group_keys = set(
        spark.table(PROGRESS_TABLE)
        .where("stage = 'hybrid_classify' and status = 'done'")
        .select("cate_1_depth", "cate_2_depth", "sc_measurement")
        .distinct()
        .rdd.map(lambda r: (r[0], r[1], int(r[2])))
        .collect()
    )
else:
    done_group_keys = set()

print(f"[RESUME] already done groups = {len(done_group_keys)}")


# --------------------------------------------------
# 1. 대상 그룹 반복
# --------------------------------------------------
for g in target_groups:
    cate1 = g["cate_1_depth"]
    cate2 = g["cate_2_depth"]
    sc = int(g["sc_measurement"])
    group_key = (cate1, cate2, sc)

    # ✅ 이미 완료된 그룹은 skip
    if group_key in done_group_keys:
        print(f"[SKIP] already done: {group_key}")
        continue

    print(f"[START] {cate1} / {cate2} / {sc}")
    group_start_ts = time.time()

    try:
        # --------------------------------------------------
        # 2. 샘플 로드
        # --------------------------------------------------
        sample_df = load_group_sample_df(
            cate1, cate2, sc, max_rows=1000
        ).cache()

        if sample_df.count() == 0:
            log_progress(
                cate1, cate2, sc,
                stage="hybrid_classify",
                status="done",
                message="no sample rows"
            )
            sample_df.unpersist()
            continue

        # --------------------------------------------------
        # 3. topic pool 로드
        # --------------------------------------------------
        topic_pool_rows = (
            spark.table(TOPIC_POOL_TABLE)
            .where(
                (F.col("cate_1_depth") == cate1) &
                (F.col("cate_2_depth") == cate2) &
                (F.col("sc_measurement") == sc)
            )
            .select("topic", "representative_memos_json")
            .collect()
        )

        results = []

        # --------------------------------------------------
        # 4. 메모 단위 분류
        # --------------------------------------------------
        for r in sample_df.collect():
            memo = r["memo"]

            # (A) overall 복귀 판단 (유사도 0.9 이상)
            overall_hit = None
            for ov in overall_rep_map.get(group_key, []):
                if match_overall_by_similarity(memo, [ov]) == ov["merged_topic"]:
                    overall_hit = ov["merged_topic"]
                    break

            if overall_hit:
                results.append((cate1, cate2, sc, overall_hit, memo))
                continue

            # (B) 일반 세부 topic 유사도
            topic = match_non_overall_by_similarity(
                memo,
                non_overall_rep_map.get(group_key, [])
            )
            if topic:
                results.append((cate1, cate2, sc, topic, memo))
                continue

            # (C) fallback LLM
            topic = classify_with_full_llm(
                memo,
                [{"topic": t["topic"]} for t in topic_pool_rows]
            )
            results.append((cate1, cate2, sc, topic, memo))

        # --------------------------------------------------
        # 5. 그룹 단위 append 저장
        # --------------------------------------------------
        result_df = spark.createDataFrame(
            results,
            ["cate_1_depth", "cate_2_depth", "sc_measurement", "topic", "memo"]
        )

        result_df.write.mode("append").format("delta").saveAsTable(DETAIL_TABLE)

        # --------------------------------------------------
        # 6. progress 기록 (✅ checkpoint)
        # --------------------------------------------------
        elapsed = round(time.time() - group_start_ts, 1)
        log_progress(
            cate1, cate2, sc,
            stage="hybrid_classify",
            status="done",
            message=f"rows={len(results)}, elapsed_sec={elapsed}"
        )

        sample_df.unpersist()
        print(f"[DONE] {group_key} ({elapsed}s)")

    except Exception as e:
        # --------------------------------------------------
        # ❌ 실패해도 다음 그룹 계속
        # --------------------------------------------------
        log_progress(
            cate1, cate2, sc,
            stage="hybrid_classify",
            status="failed",
            message=str(e)[:500]
        )
        sample_df.unpersist()
        print(f"[FAILED] {group_key} - {e}")